# 02 ? Parsers and Structural Addresses

This notebook turns raw document formats into `(m, d, p, s)` addresses for the HiSE pipeline.

In [ ]:
# Section 0: setup
%pip install -q pandas numpy
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
for candidate in [repo_root, repo_root.parent, repo_root / 'hise', repo_root.parent / 'hise']:
    if candidate.exists():
        src = candidate / 'src' if candidate.name != 'src' else candidate
        if src.exists():
            sys.path.insert(0, str(src))
        hise_src = candidate / 'hise' / 'src' if (candidate / 'hise').exists() else None
        if hise_src is not None and hise_src.exists():
            sys.path.insert(0, str(hise_src))
print('repo_root =', repo_root)
import pandas as pd
print('pandas', pd.__version__)


In [ ]:
# Section 1: Markdown parser
from hise.parsers import parse_markdown

markdown = '''
# Annual Report
## Executive Summary
Revenue grew 47%.
## Risk Factors
Customer concentration risk is material.
'''
items = parse_markdown(markdown)
print(items)
for text, address in items:
    print(text, address.as_tuple())


In [ ]:
# Section 2: Table parser
from hise.parsers import parse_table

rows = [
    ['metric', 'value'],
    ['revenue', '47%'],
    ['margin', '18%'],
]
items = parse_table(rows)
print(items)
for text, address in items:
    print(text, address.as_tuple())


In [ ]:
# Section 3: JSON parser
from hise.parsers import parse_json

obj = {
    'report': {
        'summary': {'revenue': '47%'},
        'risks': {'customer_concentration': 'important'}
    }
}
items = parse_json(obj)
print(items)
for text, addr in items:
    print(text, addr.as_tuple())


In [ ]:
# Section 4: HTML parser
from hise.parsers import parse_html

html = '<section><h2>Risks</h2><p>Revenue grew 47%</p></section>'
items = parse_html(html)
print(items)
for text, addr in items:
    print(text, addr.as_tuple())


In [ ]:
# Section 5: validation assertions
from hise.parsers import parse_markdown, parse_table, parse_json, parse_html

markdown_items = parse_markdown('# Report\n## Risks\nRevenue grew 47%')
assert len(markdown_items) == 3
assert markdown_items[1][1].depth == 2

table_items = parse_table([['metric', 'value'], ['revenue', '47%']])
assert len(table_items) == 4
assert table_items[0][1].sibling_index == 0

json_items = parse_json({'risk': {'revenue': 'concentration'}})
assert json_items[0][0] == 'concentration'

html_items = parse_html('<section><h2>Risks</h2><p>Revenue increased</p></section>')
assert len(html_items) == 2
print('parser validation passed')


## Output contract

Every parser emits a list of `(text, StructuralAddress)` pairs. The address tuple follows `(m, d, p, s)`, so downstream model code can inject structure consistently across text, table, JSON, and HTML sources.
